In [20]:
import pandas as pd
import numpy as np
from collections import defaultdict
import json

In [21]:
clues = pd.read_csv("clues_clean.csv").drop("Unnamed: 0", axis=1).reset_index()

In [22]:
clues.sample(10)

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,clue_pct
22573,22573,RACER,Dash competitor,Go-cart,"Jockey, e.g.",Marathon contender,NASCAR driver,NASCAR driver e.g.,"One making a pit stop, maybe",One who sits in laps?,Toy on a track,Track competitor,0.906114
21712,21712,PLATTER,*Serving dish,"45, e.g.",Diner offering,It may be uncovered before use,LP,Old-time record,Part of a caterer's display,Pupu ___,Something that turns on a turntable,Special dinner order,0.283351
13941,13941,INSO,I love you _____ many ways,Not ___ many words,Not ___ many words,Preceder of many words?,Start of many words?,Two words before many words,Words before many words?,___ many words,___ many words,___ many words,0.511913
15413,15413,LASER,Cutting light,Cutting-edge technology?,Eye surgery tool,High-tech surgery tool,Hologram creator,Kind of printer,Sci-fi beam,Star Wars weapon,Surgical beam,Surgical beam,0.963247
15867,15867,LETOFF,Acquit,Acquit,Acquitted,Allowed to leave the bus,Excuse from punishment,Excuse from punishment,Excuse from responsibility,Freed,Give a reprieve,Release,0.065925
847,847,AIMS,Aligns crosshairs,Dreams,Ends,Goals,Goals,Objectives,Prepares to hit a cue ball,Prepares to shoot a basket,Prepares to throw,Uses one's sight?,0.774910
19584,19584,OHOK,"Ah, got it","Ah, gotcha","Ah, now I see",Duh! Now I get it!,Now I get it,That makes sense now,"Um, sure","Well, I guess so","Well, all right","Well, all right then",0.243502
18845,18845,NIX,Invalidate,Overrule,Prohibit,Put the kibosh on,Shoot down,Shoot down,Shoot down,Turn thumbs-down on,Veto,Veto,0.542403
24096,24096,RONEE,1975 Oscar nominee Blakley,Actress Blakley,Actress Blakley,Actress Blakley,Actress Blakley,"Actress Blakley of ""Nashville""",Blakely of Nashville,"Blakley of ""Lightning Over Water""","Blakley of ""Nashville""","Blakley of ""Nashville""",0.193039
31711,31711,WHERE,I don't see it,Invitation heading,Invitation information,One of the Five W's,One of the Five W's,Question in a shell game,Seeker's question,Word in a geography quiz,___ am I?,_____ Did Our Love Go (Supremes tune),0.767117


In [23]:
clues["length"]=[len(x) for x in clues.answer]

In [24]:
clues_by_length = {l:t for l,t in clues.groupby("length")}

In [25]:
clues_by_length[3]

,index,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,clue_pct,length
0,0,AAA,Bumper-sticker letters,Grp. that offers International Driving Permits,Minor-league classification,Motorist's org.,Org. with towers,Roadside assistance grp.,Roadside assistance org.,Small battery,"Strong, as a bond",Top bond rating,0.893993,3
4,4,AAH,Circus exclamation,Foot rub reaction,Ooh and ___,Open wide sound,Reaction to a cold drink on a hot day,Sound of wonderment,That hit the spot!,That's nice,What a relief!,Word to a doctor,0.820230,3
13,13,AAR,Interlaken river,River flowing into the Rhine,River in Switzerland,Swiss river,Swiss river,Swiss river,Swiss river,Swiss river,Swiss river into the Rhine,Swiss river to the Rhine,0.522527,3
19,19,AAS,Batteries for some remotes,Furby batteries,Remote batteries,Small batteries,Small batteries,Smallish batteries,"Smoke detector batteries, often",Some batteries,Some remote power sources,___ in Able,0.678004,3
20,20,AAU,Amateur sports org.,Jr. Olympics sponsor,Junior Olympics org.,Org. that awards the Sullivan Award,Org. that bestows the Sullivan Award,"Org. with the Sullivan Award for character, le...","Org. with the Sullivan Award for character, le...",Org. with the annual Junior Olympic Games,"Sports For All, Forever org.",Sports org. for nonprofessionals,0.166078,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32443,32443,ZIT,Bit of acne,Clearasil target,Clearasil target,Minor blemish,Pimple,Proactiv target,Single outbreak,Teen's woe,"Temple spot, say?","Unwanted blemish, informally",0.442580,3
32446,32446,ZOA,Animal group suffix,Animal group suffix,"Animals, in taxonomy",Animals: Suffix,Animals: Suffix,"Combining form meaning ""animals""",Ending with proto-,Organisms: Suffix,Suffix with proto-,Taxonomy suffix,0.071555,3
32448,32448,ZOE,Actress Caldwell,Actress/model Kravitz,"Kravitz of ""Big Little Lies""",Muppet in a tutu,Playwright Akins,Pulitzer playwright Akins,Pulitzer winner Akins,"Saldana of ""Avatar""","Saldana of ""Guardians of the Galaxy""",Saldana who plays Uhura,0.477473,3
32456,32456,ZOO,Chaotic place,"Dr. Seuss' ""If I Ran the ___""","Madhouse, so to speak",Menagerie,Morning ___ (radio format),Petting setting,Place to put stock in?,Place to see pandas and panthers,Place with moving exhibits,San Diego attraction,0.641343,3


In [26]:
alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [27]:
character_set = list(alphabet)+[alphabet[i]+alphabet[j] for i in range(26) for j in range(i,26)]

In [28]:
len(character_set)

377

In [29]:
character_set[-1]

'ZZ'

In [30]:
char_table = {ch:n for n,ch in enumerate(character_set)}
idx_table = {n:ch for n,ch in enumerate(character_set)}

In [31]:
def decompose_word(w):
    if len(w) == 0:
        return []
    elif len(w) == 1:
        return [[w]]
    elif len(w) == 2:
        return [[w[0],w[1]],["".join(sorted(w))]]
    else:
        out = []
        first = w[0]
        first_suffix = decompose_word(w[1:])
        for i in first_suffix:
            out.append([first]+i)
        pair = "".join(sorted(w[:2]))
        second_suffix = decompose_word(w[2:])
        for i in second_suffix:
            out.append([pair]+i)
        return out

In [32]:
def decomp_to_idx(l):
    return tuple(char_table[i] for i in l)

In [33]:
decompose_word("ZYDECO")

[['Z', 'Y', 'D', 'E', 'C', 'O'],
 ['Z', 'Y', 'D', 'E', 'CO'],
 ['Z', 'Y', 'D', 'CE', 'O'],
 ['Z', 'Y', 'DE', 'C', 'O'],
 ['Z', 'Y', 'DE', 'CO'],
 ['Z', 'DY', 'E', 'C', 'O'],
 ['Z', 'DY', 'E', 'CO'],
 ['Z', 'DY', 'CE', 'O'],
 ['YZ', 'D', 'E', 'C', 'O'],
 ['YZ', 'D', 'E', 'CO'],
 ['YZ', 'D', 'CE', 'O'],
 ['YZ', 'DE', 'C', 'O'],
 ['YZ', 'DE', 'CO']]

In [15]:
[decomp_to_idx(x) for x in decompose_word("ZYDECO")]

[(25, 24, 3, 4, 2, 14),
 (25, 24, 3, 4, 89),
 (25, 24, 3, 79, 14),
 (25, 24, 102, 2, 14),
 (25, 24, 102, 89),
 (25, 122, 4, 2, 14),
 (25, 122, 4, 89),
 (25, 122, 79, 14),
 (375, 3, 4, 2, 14),
 (375, 3, 4, 89),
 (375, 3, 79, 14),
 (375, 102, 2, 14),
 (375, 102, 89)]

In [34]:
tags_to_word_index =[]
tags_by_length = defaultdict(list)
checked_tags = set()
for i, w, p in zip(clues["index"], clues["answer"], clues["clue_pct"]):
    d = decompose_word(w)
    for t in d:
        tags = decomp_to_idx(t)
        tags_to_word_index.append((i,tags))
        tags_by_length[len(tags)].append((tags,p))
        

In [35]:
len(tags_by_length)

14

In [38]:
for l in tags_by_length.keys():
    with open(f"data/tags_length_{l}.json", "w") as f:
        json.dump(tags_by_length[l],f)

In [40]:
with open("data/tags_to_word_index.json","w") as f:
    json.dump(tags_to_word_index, f)